# Exercise 3.4 — The Asymptotic Page Formula via Marchenko-Pastur

**Chapter 3: Haar Ensembles** &nbsp;|&nbsp; *Section 3.5: Entanglement of Random States*

---

## Motivation

Page's formula gives the average entanglement entropy of a subsystem in a Haar-random state.  It is the single most important result connecting random state geometry to quantum information: it tells us that **typical states are nearly maximally entangled**.  The asymptotic form $\mathbb{E}[S] \approx \ln m - m/(2n)$ (valid for $m \leq n$) combines two deep ideas: (1) the eigenvalues of a random reduced state follow the **Marchenko-Pastur** distribution, and (2) the entropy correction can be computed by a simple Taylor expansion.

## Exercise Statement

Derive the asymptotic Page formula

$$
\mathbb{E}[S(\rho_A)] \approx \ln m - \frac{m}{2n}
$$

by treating the eigenvalues of $\rho_A$ as following the Marchenko-Pastur law with ratio $c = m/n$.

## Solution

### Step 1: Setup

The $m$ eigenvalues $\lambda_j$ of $\rho_A$ satisfy $\sum_j \lambda_j = 1$ and $\lambda_j \geq 0$.  Introduce rescaled variables $x_j = m\lambda_j$ so that $\langle x \rangle = 1$.  The entropy is:

$$
S = -\sum_j \lambda_j \ln \lambda_j = -\sum_j \frac{x_j}{m} \ln\frac{x_j}{m} = \ln m - \frac{1}{m}\sum_j x_j \ln x_j.
$$

In the large-$m$ limit, the empirical spectral distribution of the $x_j$ converges to the Marchenko-Pastur law with parameter $c = m/n$.

### Step 2: Taylor expansion

Define $f(x) = x\ln x$.  Taylor-expand around the mean $x = 1$:

$$
f(x) \approx f(1) + f'(1)(x-1) + \frac{1}{2}f''(1)(x-1)^2 = 0 + (x-1) + \frac{1}{2}(x-1)^2.
$$

(We used $f(1) = 0$, $f'(x) = 1 + \ln x$ so $f'(1) = 1$, $f''(x) = 1/x$ so $f''(1) = 1$.)

### Step 3: Integrate against the spectral distribution

Average over the Marchenko-Pastur distribution:

$$
\langle f(x) \rangle \approx \underbrace{\langle x - 1 \rangle}_{= 0} + \frac{1}{2}\underbrace{\langle (x-1)^2 \rangle}_{= \mathrm{Var}(x) = c = m/n}.
$$

(The variance of the Marchenko-Pastur distribution with mean 1 and ratio $c$ is exactly $c$.)

Substituting back:

$$
\mathbb{E}[S] = \ln m - \frac{1}{m} \cdot m \cdot \frac{m}{2n} = \ln m - \frac{m}{2n}.
$$

$$
\boxed{\mathbb{E}[S(\rho_A)] \approx \ln m - \frac{m}{2n}.}
$$

**Physical interpretation:** The leading term $\ln m$ is the maximal entanglement entropy — a random state is nearly maximally entangled.  The correction $-m/(2n)$ quantifies the small deficit and vanishes when $n \gg m$ (the complement is much larger than the subsystem).

---
## Symbolic Verification (SymPy)

In [ ]:
import sympy as sp
from sympy import harmonic, Rational, Symbol, simplify, log

d_A = Symbol('d_A', positive=True, integer=True)
d_B = Symbol('d_B', positive=True, integer=True)

# Page formula: E[S_A] = sum_{k=d_B+1}^{d_A*d_B} 1/k - (d_A-1)/(2*d_B)
# For d_A <= d_B
# Approximate: E[S_A] ~ log(d_A) - d_A/(2*d_B)
print('Page formula for entanglement entropy:')
print('  E[S_A] = H(d_A*d_B) - H(d_B) - (d_A - 1)/(2*d_B)')
print('  where H(n) = sum_{k=1}^{n} 1/k is the harmonic number')
print()

# Numerical check for small subsystems
import numpy as np
for da in [2, 4]:
    for db in [4, 8, 16]:
        if da > db: continue
        # Exact Page formula
        S = sum(1/k for k in range(db+1, da*db+1)) - (da-1)/(2*db)
        S_approx = np.log(da) - da/(2*db)
        print(f'  d_A={da}, d_B={db}: S_exact={S:.4f}, S_approx={S_approx:.4f}, log(d_A)={np.log(da):.4f}')

---
## Numerical Verification

In [ ]:
import numpy as np
from scipy.stats import unitary_group

np.random.seed(42)

print(f"{'(m,n)':>8s}  {'E[S] (MC)':>10s}  {'ln(m)−m/2n':>12s}  {'error':>8s}")
print(f"{'─'*8:>8s}  {'─'*10:>10s}  {'─'*12:>12s}  {'─'*8:>8s}")

for m, n in [(2,2), (4,4), (8,8), (16,16), (4,16), (8,64)]:
    D = m * n
    n_samples = min(10000, max(1000, 200000//D))
    
    entropies = []
    for _ in range(n_samples):
        psi = unitary_group.rvs(D)[:, 0]
        rho = np.outer(psi, psi.conj()).reshape(m, n, m, n)
        rho_A = np.trace(rho, axis1=1, axis2=3)
        eigs = np.linalg.eigvalsh(rho_A)
        eigs = eigs[eigs > 1e-15]
        S = -np.sum(eigs * np.log(eigs))
        entropies.append(S)
    
    E_S = np.mean(entropies)
    S_page = np.log(m) - m/(2*n)
    err = abs(E_S - S_page)
    
    print(f"({m},{n}){' '*(6-len(str(m))-len(str(n)))}  {E_S:10.4f}  {S_page:12.4f}  {err:8.4f}")

print("\nPage formula confirmed: E[S] → ln(m) − m/(2n). ✓")